# NeuroSim — Notebook 02: Minimum Control Energy
## Finite-Horizon State Transition Analysis

**What this notebook demonstrates:**
1. Why the finite horizon T is a physiological constant, not a free parameter
2. How control energy depends on structural connectome topology
3. The minimum-energy optimal control trajectory between brain states
4. Critical comparison: finite-horizon vs infinite-horizon energy estimates

**Data:** Synthetic 20-region connectome (ring + random structure).
*On June 9: swap Section 1 for your HCP Schaefer-400 matrices — everything else runs unchanged.*

**References:**
- Gu et al. (2015) Nature Comms — Average/modal controllability
- Srivastava et al. (2020) PLOS Comp. Biol. — Finite-horizon NCT
- Van Loan (1978) IEEE TAC — Doubling Algorithm

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.linalg import solve_discrete_lyapunov
from numpy.linalg import svd

from neurosim.physics import (
    normalise_matrix,
    compute_gramian_doubling,
    minimum_energy,
    average_controllability,
    modal_controllability,
)
from neurosim.connectivity import ridge_effective_connectivity

plt.rcParams.update({
    'figure.dpi': 150,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
    'axes.titlesize': 12,
    'lines.linewidth': 2,
})

BLUE   = '#2E6DA4'
RED    = '#C0392B'
GREEN  = '#27AE60'
ORANGE = '#E67E22'
GREY   = '#7F8C8D'

print("NeuroSim Notebook 02: Minimum Control Energy")
print(f"NumPy {np.__version__}")

## Section 1: Data — [SYNTHETIC]
### Replace this section with real HCP data on June 9

Builds a synthetic N=20 region connectome:
- **Ring backbone:** each region connected to its two neighbours
- **Sparse long-range connections:** 15% random density
- **MVAR BOLD:** time series driven by the connectome

**To swap in real data, replace cells 2–3 with:**
```python
from neurosim.loader import load_connectome, from_arrays
SC = load_connectome('path/to/sub-01_SC_schaefer400.npy')
X  = np.load('path/to/sub-01_BOLD_schaefer400.npy')  # (400, 1200)
N  = SC.shape[0]
```

In [ ]:
# ── SYNTHETIC DATA BLOCK — replace with real HCP matrices on June 9 ────────
rng = np.random.default_rng(42)
N   = 20     # regions  (Schaefer-400 uses 400 — same code, bigger matrices)
T   = 1200   # timepoints (HCP: TR=720ms, 14.4 min resting-state)

# Structural connectome: ring backbone + sparse long-range connections
SC = np.zeros((N, N))
for i in range(N):
    SC[i, (i+1) % N] = 1.0
    SC[(i+1) % N, i] = 1.0
    if rng.random() < 0.15:
        j = rng.integers(0, N)
        SC[i, j] = rng.uniform(0.3, 0.8)
        SC[j, i] = SC[i, j]
np.fill_diagonal(SC, 0)
SC = (SC + SC.T) / 2.0

# Simulate BOLD-like time series driven by SC structure
A_gen = normalise_matrix(SC + 0.05*rng.normal(0,1,(N,N)), target_rho=0.85)
X = np.zeros((N, T))
for t in range(1, T):
    X[:, t] = A_gen @ X[:, t-1] + rng.normal(0, 0.3, N)

# Z-score (standard fMRI preprocessing)
X = (X - X.mean(axis=1, keepdims=True)) / (X.std(axis=1, keepdims=True) + 1e-8)

print(f"Connectome: {N} regions, {(SC > 0).sum()//2} edges")
print(f"SC density: {(SC > 0).mean()*100:.1f}%")
print(f"BOLD: ({N}, {T})")

## Section 2: Effective Connectivity Estimation

We estimate the system operator **A** via MVAR Ridge regression.
The key distinction from standard NCT: asymmetric EC, not symmetric FC.
This preserves causal directionality and avoids the Teleportation Error.

In [ ]:
EC = ridge_effective_connectivity(X, alpha=1.0, lag=1)
A  = normalise_matrix(EC, target_rho=0.9)
B  = np.eye(N)  # full-rank input

print(f"EC: shape={EC.shape}, asymmetric={not np.allclose(EC, EC.T)}")
print(f"Spectral radius after normalisation: {np.max(np.abs(np.linalg.eigvals(A))):.6f}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
im0 = axes[0].imshow(SC, cmap='Blues', aspect='auto')
axes[0].set_title('Structural Connectome (SC)')
axes[0].set_xlabel('Region'); axes[0].set_ylabel('Region')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(EC, cmap='RdBu_r', aspect='auto',
                      vmin=-np.abs(EC).max(), vmax=np.abs(EC).max())
axes[1].set_title('Effective Connectivity (EC)\nRidge MVAR, asymmetric')
axes[1].set_xlabel('Region (source)'); axes[1].set_ylabel('Region (target)')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

sc_vals = SC[np.triu_indices(N, k=1)]
ec_vals = np.abs(EC + EC.T)[np.triu_indices(N, k=1)] / 2
axes[2].scatter(sc_vals, ec_vals, alpha=0.6, color=BLUE, s=30)
axes[2].set_xlabel('SC weight'); axes[2].set_ylabel('|EC| (symmetrised)')
axes[2].set_title('Structure-Function Coupling')
corr = np.corrcoef(sc_vals, ec_vals)[0,1]
axes[2].text(0.05, 0.92, f'r = {corr:.3f}', transform=axes[2].transAxes, color=BLUE)

plt.tight_layout()
plt.savefig('sc_ec_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"SC-EC correlation: r = {corr:.3f}")

## Section 3: Controllability Landscape

- **Average controllability:** how easily a node steers to many states (W∞ diagonal). High in connector hubs.
- **Modal controllability:** ability to reach hard-to-reach states. High in peripheral regions.

These are complementary — average captures *breadth*, modal captures *difficulty*.

In [ ]:
ac = average_controllability(A)
mc = modal_controllability(A)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(range(N), ac, color=BLUE, alpha=0.85, edgecolor='white')
axes[0].set_title('Average Controllability\n(diagonal of W∞)')
axes[0].set_xlabel('Region'); axes[0].set_ylabel('Average controllability')
axes[0].axhline(ac.mean(), color=RED, ls='--', lw=1.5, label=f'Mean = {ac.mean():.3f}')
axes[0].legend(fontsize=9)

axes[1].bar(range(N), mc, color=ORANGE, alpha=0.85, edgecolor='white')
axes[1].set_title('Modal Controllability\n(hard-to-reach state access)')
axes[1].set_xlabel('Region'); axes[1].set_ylabel('Modal controllability')
axes[1].axhline(mc.mean(), color=RED, ls='--', lw=1.5, label=f'Mean = {mc.mean():.3f}')
axes[1].legend(fontsize=9)

axes[2].scatter(ac, mc, c=range(N), cmap='viridis', s=60, zorder=3)
axes[2].set_xlabel('Average controllability')
axes[2].set_ylabel('Modal controllability')
axes[2].set_title('Average vs Modal\n(network trade-off)')
corr_ac_mc = np.corrcoef(ac, mc)[0,1]
axes[2].text(0.05, 0.92, f'r = {corr_ac_mc:.3f}', transform=axes[2].transAxes, color=GREY)
for i in range(N):
    axes[2].annotate(str(i), (ac[i], mc[i]), fontsize=7, alpha=0.7)

plt.tight_layout()
plt.savefig('controllability_landscape.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"AC: mean={ac.mean():.4f}, max=Node {ac.argmax()} ({ac.max():.4f})")
print(f"MC: mean={mc.mean():.4f}, max=Node {mc.argmax()} ({mc.max():.4f})")
print(f"AC-MC correlation: r = {corr_ac_mc:.3f} (expected negative for hub-periphery networks)")

## Section 4: Finite-Horizon Gramian — Horizon Sweep

We sweep T = 1 to 30 TRs (~0.7s to ~21.6s at HCP TR=720ms).

The diagonal of W_T gives **finite-horizon average controllability** — how reachable
each region is within exactly T steps. This replaces the biologically indefensible W∞.

In [ ]:
horizons = list(range(1, 31))
gramian_diags = np.array([np.diag(compute_gramian_doubling(A, B, T=T)) for T in horizons])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

im = axes[0].imshow(gramian_diags.T, aspect='auto', cmap='viridis',
                     extent=[horizons[0], horizons[-1], N-0.5, -0.5])
axes[0].set_xlabel('Horizon T (TR steps)'); axes[0].set_ylabel('Region')
axes[0].set_title('Finite-Horizon Controllability\nW_T diagonal across horizons')
plt.colorbar(im, ax=axes[0], fraction=0.046, label='Reachability')

mean_r = gramian_diags.mean(axis=1)
axes[1].plot(horizons, mean_r, color=BLUE, lw=2.5)
axes[1].fill_between(horizons, gramian_diags.min(axis=1), gramian_diags.max(axis=1),
                      alpha=0.2, color=BLUE, label='Min-max range')
axes[1].set_xlabel('Horizon T (TR steps)')
axes[1].set_ylabel('Mean reachability')
axes[1].set_title('Mean Network Reachability vs T')
for t_label, t_val, col in [('1s',1,GREY), ('5s',7,ORANGE), ('10s',14,RED)]:
    axes[1].axvline(t_val, color=col, ls='--', alpha=0.7, lw=1.5)
    axes[1].text(t_val+0.3, mean_r[t_val-1]*0.95, t_label, color=col, fontsize=9)
axes[1].legend(fontsize=9)

for T_show, col, label in [(1,GREEN,'T=1 (~0.7s)'), (10,BLUE,'T=10 (~7.2s)'), (30,RED,'T=30 (~21.6s)')]:
    axes[2].plot(gramian_diags[horizons.index(T_show)], color=col, label=label, lw=2)
axes[2].set_xlabel('Region'); axes[2].set_ylabel('Reachability (W_T diagonal)')
axes[2].set_title('Controllability Profile at\nDifferent Horizons')
axes[2].legend(fontsize=9)

plt.suptitle('Finite-Horizon Reachability Gramian — Van Loan Doubling', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('gramian_horizon_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Most reachable at T=10: Node {gramian_diags[9].argmax()} ({gramian_diags[9].max():.4f})")

## Section 5: Minimum Control Energy — State Transitions

**The clinical core of NeuroSim.**

$$E^* = (x_T - A^T x_0)^\top W_T^{-1} (x_T - A^T x_0)$$

This energy is a biomarker. In AUD, the energy to exit the craving attractor is elevated.
In Alzheimer's, energy to access memory circuits increases with disease progression.

In [ ]:
# Define brain states from BOLD
# In real HCP data: use task block onsets to define x_rest and x_task
x_rest = X[:, :T//3].mean(axis=1)
global_signal = X.mean(axis=0)
x_task = X[:, global_signal > np.percentile(global_signal, 75)].mean(axis=1)
x_low  = X[:, global_signal < np.percentile(global_signal, 25)].mean(axis=1)

# Normalise to unit norm (standard NCT convention)
x_rest = x_rest / (np.linalg.norm(x_rest) + 1e-8)
x_task = x_task / (np.linalg.norm(x_task) + 1e-8)
x_low  = x_low  / (np.linalg.norm(x_low)  + 1e-8)

print("Brain states defined:")
print(f"  x_rest norm={np.linalg.norm(x_rest):.4f}")
print(f"  x_task norm={np.linalg.norm(x_task):.4f}")
print(f"  rest-task angular distance: {np.arccos(np.clip(x_rest @ x_task,-1,1))*180/np.pi:.1f}°")

In [ ]:
T_range = list(range(1, 25))
transitions = {
    'Rest → Task': (x_rest, x_task, BLUE),
    'Task → Rest': (x_task, x_rest, GREEN),
    'Rest → Low':  (x_rest, x_low,  ORANGE),
}
energy_vs_T = {name: [minimum_energy(A, B, x0, xT, T=T)[0]
                       for T in T_range]
               for name, (x0, xT, _) in transitions.items()}

T_primary = 10
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for name, (x0, xT, col) in transitions.items():
    axes[0].semilogy(T_range, energy_vs_T[name], color=col, label=name, lw=2)
axes[0].axvline(T_primary, color=GREY, ls='--', lw=1.5, label=f'T={T_primary}')
axes[0].set_xlabel('Horizon T (TR steps)')
axes[0].set_ylabel('Minimum Control Energy E* (log scale)')
axes[0].set_title('Control Energy vs Horizon T')
axes[0].legend(fontsize=9)

e_rt, u_opt_rt = minimum_energy(A, B, x_rest, x_task, T=T_primary)
axes[1].bar(range(N), u_opt_rt,
            color=[GREEN if u > 0 else RED for u in u_opt_rt],
            alpha=0.85, edgecolor='white')
axes[1].axhline(0, color=GREY, lw=0.8)
axes[1].set_xlabel('Region'); axes[1].set_ylabel('Optimal control input u*(0)')
axes[1].set_title(f'Optimal First-Step Control\nRest → Task (T={T_primary})')
axes[1].text(0.05, 0.95, f'E* = {e_rt:.4f}', transform=axes[1].transAxes,
             fontsize=10, color=BLUE, va='top')

# State space: PCA projection
all_states = np.column_stack([x_rest, x_task, x_low])
U, S, Vt = svd(all_states - all_states.mean(axis=1, keepdims=True))
proj = U[:, :2].T
A_T_mat = np.linalg.matrix_power(A, T_primary)
x_free = A_T_mat @ x_rest

for label, vec, marker, col, ms in [
    ('Rest (x₀)', x_rest, 'o', BLUE, 12),
    ('Task (xT)', x_task, '*', GREEN, 14),
    ('Free evol.', x_free, 's', GREY, 10),
    ('Low act.', x_low, 'D', ORANGE, 9),
]:
    xy = proj @ vec
    axes[2].scatter(*xy, marker=marker, color=col, s=ms**2, label=label, zorder=5)

r2d = proj @ x_rest; t2d = proj @ x_task
axes[2].annotate('', xy=t2d, xytext=r2d,
                 arrowprops=dict(arrowstyle='->', color=BLUE, lw=1.5))
axes[2].set_xlabel('PC 1'); axes[2].set_ylabel('PC 2')
axes[2].set_title('Brain State Space (PCA)\nOptimal control transition')
axes[2].legend(fontsize=8)

plt.suptitle('Minimum Control Energy — NeuroSim Finite-Horizon Physics Engine',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('minimum_energy_transitions.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"E* at T={T_primary}:")
for name, (x0, xT, _) in transitions.items():
    e, _ = minimum_energy(A, B, x0, xT, T=T_primary)
    print(f"  {name}: {e:.6f}")

## Section 6: The Vanishing Cost Problem

As T → ∞, minimum control energy collapses to zero for any stable system.
This is why the infinite-horizon Gramian W∞ is biologically indefensible:
it assumes the brain has unlimited time to transition between states.

**The finite-horizon metric is the only clinically meaningful one.**

In [ ]:
T_extended = list(range(1, 100, 2))
energy_finite = [minimum_energy(A, B, x_rest, x_task, T=T)[0] for T in T_extended]

W_inf = solve_discrete_lyapunov(A, B @ B.T)
W_inf_inv = np.linalg.pinv(W_inf)
delta_inf  = x_task - np.linalg.matrix_power(A, 200) @ x_rest
energy_inf = float(delta_inf @ W_inf_inv @ delta_inf)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].semilogy(T_extended, energy_finite, color=BLUE, lw=2.5,
                  label='Finite-horizon E*(T)')
axes[0].axhline(energy_inf, color=RED, ls='--', lw=2,
                label=f'Infinite-horizon ≈ {energy_inf:.2e}')
axes[0].axvline(10, color=GREY, ls=':', lw=1.5, label='T=10 (cognitive window)')
axes[0].set_xlabel('Horizon T (TR steps)')
axes[0].set_ylabel('Minimum Control Energy (log scale)')
axes[0].set_title('The Vanishing Cost Problem\nWhy infinite-horizon NCT fails')
axes[0].legend(fontsize=9)
axes[0].annotate('Energy → 0\nas T → ∞',
                 xy=(T_extended[-1], energy_finite[-1]),
                 xytext=(50, energy_finite[10]),
                 fontsize=9, color=RED,
                 arrowprops=dict(arrowstyle='->', color=RED, lw=1.2))

ratio = [e / (energy_inf + 1e-12) for e in energy_finite]
axes[1].plot(T_extended, ratio, color=ORANGE, lw=2.5)
axes[1].axvline(10, color=GREY, ls=':', lw=1.5, label='T=10')
axes[1].axhline(1.0, color=RED, ls='--', lw=1, alpha=0.5, label='Infinite baseline')
axes[1].set_xlabel('Horizon T (TR steps)')
axes[1].set_ylabel('E*(T) / E*∞')
axes[1].set_title('Finite/Infinite Energy Ratio\nClinical sensitivity amplification')
axes[1].legend(fontsize=9)
t10_idx = T_extended.index(9)  # T=9 is closest to T=10 in step-2 range
axes[1].text(0.97, 0.97,
             f'T=10: finite = {energy_finite[t10_idx]:.4f}\n'
             f'T=∞:  approx = {energy_inf:.2e}\n'
             f'Ratio = {ratio[t10_idx]:.0f}×',
             transform=axes[1].transAxes, fontsize=9, va='top', ha='right',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Finite-Horizon vs Infinite-Horizon NCT — The Clinical Argument',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('vanishing_cost.png', dpi=150, bbox_inches='tight')
plt.show()

print("Vanishing cost demonstration:")
print(f"  T=1:   E* = {energy_finite[0]:.6f}")
print(f"  T=9:   E* = {energy_finite[t10_idx]:.6f}")
print(f"  T=49:  E* = {energy_finite[24]:.6f}")
print(f"  T=99:  E* = {energy_finite[-1]:.6f}")
print(f"  W∞:    E* ≈ {energy_inf:.2e}")
print(f"\nInfinite-horizon erases {ratio[t10_idx]:.0f}× clinical signal at T=10.")

## Section 7: Per-Node Stimulation Targets

**Clinical translation:** Which single region, if stimulated, requires the
least energy to drive the system from rest to task?

These are the **optimal TMS/tDCS targets**. In a real AUD discordant-twin study,
the node with the largest *increase* in E* (AUD twin vs healthy co-twin) is
the dysregulated hub — the neuromodulation candidate.

In [ ]:
T_stim = 10
energies_per_node = []
for i in range(N):
    B_i = np.zeros((N, 1)); B_i[i] = 1.0
    e, _ = minimum_energy(A, B_i, x_rest, x_task, T=T_stim)
    energies_per_node.append(e)
energies_per_node = np.array(energies_per_node)

finite_mask = np.isfinite(energies_per_node) & (energies_per_node < 1e10)
best_node   = int(np.where(finite_mask, energies_per_node, np.inf).argmin())
valid_idx   = np.where(finite_mask)[0]
sorted_idx  = valid_idx[np.argsort(energies_per_node[valid_idx])]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

display_e = np.where(finite_mask, energies_per_node,
                      energies_per_node[finite_mask].max() * 1.5)
colors = [GREEN if i == best_node else BLUE for i in range(N)]
axes[0].bar(range(N), display_e, color=colors, alpha=0.85, edgecolor='white')
axes[0].set_xlabel('Region (stimulation site)')
axes[0].set_ylabel(f'E* (Rest→Task, T={T_stim})')
axes[0].set_title('Single-Node Stimulation Energy\n(lower = better therapeutic target)')
axes[0].annotate(f'Best target\nNode {best_node}',
                 xy=(best_node, display_e[best_node]),
                 xytext=(best_node+2, display_e[best_node]*1.5),
                 fontsize=9, color=GREEN,
                 arrowprops=dict(arrowstyle='->', color=GREEN))

top_k = min(10, len(sorted_idx))
axes[1].barh(range(top_k), energies_per_node[sorted_idx[:top_k]],
             color=BLUE, alpha=0.85, edgecolor='white')
axes[1].set_yticks(range(top_k))
axes[1].set_yticklabels([f'Node {i}' for i in sorted_idx[:top_k]])
axes[1].invert_yaxis()
axes[1].set_xlabel(f'E* (Rest→Task, T={T_stim})')
axes[1].set_title(f'Top {top_k} Stimulation Targets\n(ranked by minimum energy)')
e_all, _ = minimum_energy(A, B, x_rest, x_task, T=T_stim)
axes[1].axvline(e_all, color=RED, ls='--', lw=1.5,
                label=f'Full-network E*={e_all:.3f}')
axes[1].legend(fontsize=9)

plt.suptitle('Therapeutic Target Identification via Control Energy',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('stimulation_targets.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Best stimulation target: Node {best_node} (E*={energies_per_node[best_node]:.6f})")
print(f"Full-network E* (all regions): {e_all:.6f}")
print(f"Top 5 targets: {list(sorted_idx[:5])}")

## Summary

| Step | Demonstrated | Clinical relevance |
|------|-------------|-------------------|
| 1 | 20-region synthetic connectome | Swap for HCP SC matrix June 9 |
| 2 | Directed EC via MVAR ridge | Avoids Teleportation Error |
| 3 | Average + modal controllability | Hub vs peripheral regions |
| 4 | Finite-horizon Gramian sweep (T=1–30) | Physiological timescale sensitivity |
| 5 | Minimum state transition energy | Core NCT biomarker |
| 6 | Vanishing cost problem | Why W∞ fails clinically |
| 7 | Per-node stimulation targets | Direct TMS/tDCS planning |

**To run on real HCP data:** Replace Section 1 with:
```python
SC = load_connectome('path/to/SC_schaefer400.npy')
X  = np.load('path/to/BOLD_schaefer400.npy')  # (400, 1200)
N  = SC.shape[0]
```

**Next:** Notebook 03 — Wilson-Cowan non-linear benchmark